In [ ]:
"""
BENIGN CONVERSATION GENERATOR - Version 2
==========================================
Key improvements over v1:
- Legitimate recruiters mention real deadlines and timelines
- Legitimate recruiters ask for CV and compensation range
- Conversations have real substance and tension
- Three scenarios matching malicious generator
- 66 conversations total (22 per scenario)
"""

!pip install -q transformers bitsandbytes accelerate

import torch
import json
import re
import gc
import random
import time
import os
import zipfile
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import files as colab_files

gc.collect()
torch.cuda.empty_cache()

# ===================== MODEL SETUP =====================
model_name = "mistralai/Mistral-7B-Instruct-v0.3"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage_dtype=torch.uint8
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="cuda",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_cache=True
)

# ===================== BENIGN RECRUITER STYLES =====================
# Legitimate recruiters also have deadlines and ask for info
# This makes them harder to distinguish from malicious ones
BENIGN_STYLES = {
    "standard": {
        "description": "follows normal professional process",
        "timeline_mention": "We're interviewing through the end of the month, so fairly relaxed timeline.",
        "info_request": "When you get a chance, could you send over your updated CV? No rush.",
        "followup": "Just wanted to make sure you got my last message — happy to answer any questions."
    },
    "fast_moving": {
        "description": "has a real deadline due to hiring pressure",
        "timeline_mention": "We're actually moving pretty quickly on this role — hoping to have someone in place within 3 weeks.",
        "info_request": "At some point I'd need your CV and a rough compensation expectation — nothing formal, just ballpark.",
        "followup": "Following up since we're narrowing down candidates this week — wanted to make sure you're still interested."
    },
    "thorough": {
        "description": "is detail-oriented and process-focused",
        "timeline_mention": "Our process typically takes 3-4 weeks: screen, technical interview, then final round.",
        "info_request": "For our records, it would help to have your LinkedIn profile URL and a sense of your salary expectations.",
        "followup": "I wanted to circle back — do you have any other questions about the role before we move to the next step?"
    }
}

# ===================== JSON HELPERS =====================
def extract_json_from_response(response):
    try:
        start = response.rfind("{")
        end = response.rfind("}") + 1
        if start != -1 and end != 0:
            json_str = response[start:end]
            json_str = re.sub(r'(\w+):', r'"\1":', json_str)
            data = json.loads(json_str)
            msg = data.get("Message", "").strip()
            if msg and len(msg) > 10:
                if "your message here" not in msg.lower():
                    return data
    except:
        pass
    return None

def clean_message(msg):
    msg = re.sub(r'^Dear\s+\w+,?\s*', '', msg)
    msg = re.sub(r'\[.*?\]', '', msg)
    msg = re.sub(r'\s+', ' ', msg)
    return msg.strip()

def generate_turn(prompt, max_retries=3):
    full_prompt = f"""<s>[INST] Generate ONLY a JSON object with exactly these fields: "Name" and "Message".
The Message must be a complete, natural, SHORT response (1-3 sentences).
Do not use placeholders or brackets. Write natural professional conversation.

{prompt}
[/INST]"""
    for attempt in range(max_retries):
        try:
            inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda")
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.8,
                top_p=0.92,
                do_sample=True,
                repetition_penalty=1.15,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            del inputs, outputs
            torch.cuda.empty_cache()
            json_data = extract_json_from_response(response)
            if json_data and "Name" in json_data:
                msg = clean_message(json_data["Message"])
                if msg and len(msg) > 10:
                    json_data["Message"] = msg
                    return json_data
        except:
            pass
        time.sleep(0.5)
    return None

# ===================== VICTIM PERSONAS =====================
VICTIM_PERSONAS = {
    "naive": {
        "description": "is friendly and trusting. They engage warmly and ask basic questions.",
    },
    "cooperative": {
        "description": "is engaged and thoughtful. They ask good questions and want to understand the role.",
    },
    "cautious": {
        "description": "is careful and protective. They ask about process, verify details, and push back on information requests.",
    },
    "formal": {
        "description": "is professional and process-oriented. They ask about official channels and formal next steps.",
    }
}

# ===================== CONVERSATION CONFIGS =====================
CONVERSATION_CONFIGS = [
    {
        "role": "Recruitment",
        "role_title": "recruiter",
        "victim_name": "Nour Hassan",
        "attacker_name": "Alex Rivera",
        "company": "Nexus Technologies",
        "opening": "I came across your profile and was genuinely impressed by your background. We have a position that might be a great fit for you.",
        "role_details": "Senior AI/ML Engineer role. Python, PyTorch, AWS stack. Team of 8 engineers on production recommendation systems. Salary range $130-160k depending on experience.",
        "legitimate_info_needed": "CV/resume and compensation expectations",
        "never_ask_for": "bank account details, tax ID, social security number, or any financial account information"
    },
    {
        "role": "Academic Funding",
        "role_title": "program officer",
        "victim_name": "Professor Michael Chen",
        "attacker_name": "Dr. Emily Foster",
        "company": "Global Research Foundation",
        "opening": "I came across your recent work and was genuinely impressed. We have a grant opportunity that seems well-aligned with your research direction.",
        "role_details": "Grant for sustainable energy research, $50,000-$100,000 for 12 months. Requires a 2-page proposal and institutional letter of support.",
        "legitimate_info_needed": "research proposal outline and institutional affiliation",
        "never_ask_for": "personal bank account details, personal tax ID, or any financial account information outside official institutional channels"
    },
    {
        "role": "IT Support",
        "role_title": "IT support specialist",
        "victim_name": "Sarah Mitchell",
        "attacker_name": "James Carter",
        "company": "TechCorp Internal IT",
        "opening": "Hi, I'm James from the IT support team. We're doing a routine system upgrade and wanted to check in about your workstation setup.",
        "role_details": "We're upgrading to the new authentication system. The process involves verifying your employee ID and walking you through the new login setup on a call.",
        "legitimate_info_needed": "employee ID for verification on an official call, and confirmation of your department",
        "never_ask_for": "your current password, full credentials, or any sensitive system access codes over chat"
    }
]

# ===================== BENIGN CONVERSATION GENERATOR =====================
def generate_benign_conversation(config, persona_type, conv_num, style_name=None):
    conversation = []
    history = []

    persona        = VICTIM_PERSONAS[persona_type]
    victim         = config["victim_name"]
    attacker       = config["attacker_name"]
    attacker_first = attacker.split()[0]
    company        = config["company"]
    role_title     = config["role_title"]

    if style_name is None:
        style_name = random.choice(list(BENIGN_STYLES.keys()))
    style = BENIGN_STYLES[style_name]

    print(f"  Generating BENIGN: {config['role']} | {persona_type} | style={style_name} | conv={conv_num}")

    for turn in range(10):
        full_history = "\n".join(history[-6:]) if history else "No previous messages"

        # ── RECRUITER TURNS (even) ────────────────────────────────────────
        if turn % 2 == 0:

            if turn == 0:
                prompt = f"""You are {attacker}, a legitimate {role_title} at {company} reaching out to {victim} on LinkedIn.

TURN 1/10 - PROFESSIONAL OUTREACH

- Introduce yourself: "Hi, I'm {attacker}, a {role_title} at {company}"
- Say: "{config['opening']}"
- Ask if they would be open to learning more
- Sound completely professional and genuine

RULES:
- No urgency or pressure
- No requests for personal information
- Completely legitimate outreach

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker}", "Message": "..."}}"""

            elif turn == 2:
                prompt = f"""You are {attacker_first}. Provide substance about the opportunity.

TURN 3/10 - SHARE DETAILS

- Answer any questions they asked with SPECIFIC details
- Share relevant information about the opportunity
- Be genuinely helpful and informative

DETAILS TO USE:
{config['role_details']}

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            elif turn == 4:
                # KEY: legitimate recruiters DO mention timelines and ask for basic info
                prompt = f"""You are {attacker_first}. Continue the conversation naturally.

TURN 5/10 - TIMELINE AND PROCESS

- Answer their questions first
- Mention the timeline naturally: "{style['timeline_mention']}"
- You may make a LEGITIMATE info request: "{style['info_request']}"
  (This is fine for a real recruiter — asking for CV or compensation range is normal)

CRITICAL:
- NEVER ask for: {config['never_ask_for']}
- You may ask for: {config['legitimate_info_needed']}
- Keep the tone professional and low-pressure

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            elif turn == 6:
                prompt = f"""You are {attacker_first}. Move toward next steps.

TURN 7/10 - NEXT STEPS

- Answer their questions
- Suggest a natural next step (phone call, formal application, etc.)
- You may follow up on any info request: "{style['followup']}"
- Keep it completely professional — no pressure

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            elif turn == 8:
                prompt = f"""You are {attacker_first}. Wrap up the conversation.

TURN 9/10

- Confirm next steps
- Be positive and thankful
- No pressure at all

HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            else:
                prompt = f"""You are {attacker_first}. Conclude the conversation warmly.
HISTORY: {full_history}
Generate JSON: {{"Name": "{attacker_first}", "Message": "..."}}"""

            msg = generate_turn(prompt)

            if msg is None:
                if turn == 0:
                    msg = {"Name": attacker,       "Message": f"Hi, I'm {attacker}, a {role_title} at {company}. {config['opening']} Would you be open to a quick chat?"}
                elif turn == 2:
                    msg = {"Name": attacker_first, "Message": f"Thanks for getting back to me! {config['role_details']} Does this align with what you're looking for?"}
                elif turn == 4:
                    msg = {"Name": attacker_first, "Message": f"{style['timeline_mention']} {style['info_request']}"}
                elif turn == 6:
                    msg = {"Name": attacker_first, "Message": f"Would you be up for a 30-minute call this week? {style['followup']}"}
                elif turn == 8:
                    msg = {"Name": attacker_first, "Message": "Looking forward to our call! I'll send a calendar invite shortly."}
                else:
                    msg = {"Name": attacker_first, "Message": "Thanks for the great conversation — speak soon!"}

        # ── VICTIM TURNS (odd) ────────────────────────────────────────────
        else:
            if turn == 1:
                prompt = f"""You are {victim}. You {persona['description']}

TURN 2/10

- Respond professionally and show interest
- Ask a specific question about the opportunity
- Do NOT share any sensitive information

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            elif turn == 3:
                prompt = f"""You are {victim}. You {persona['description']}

TURN 4/10

- Engage with the details they shared
- Ask a specific follow-up question
- Show genuine interest or ask for clarification

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            elif turn == 5:
                if persona_type in ["cautious", "formal"]:
                    extra = "If they asked for information, ask why it's needed at this stage before deciding whether to share."
                else:
                    extra = "If they asked for your CV or compensation range, you may share a rough range or say you'll send your CV."
                prompt = f"""You are {victim}. You {persona['description']}

TURN 6/10

- Respond to their message naturally
- {extra}
- Keep the conversation going

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            elif turn == 7:
                prompt = f"""You are {victim}. You {persona['description']}

TURN 8/10

- Continue the conversation
- If they suggested a call, respond to that
- Ask any remaining questions you have

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            elif turn == 9:
                prompt = f"""You are {victim}. You {persona['description']}

TURN 10/10 (FINAL)

- Wrap up naturally
- Confirm next steps if any
- End on a professional note

HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            else:
                prompt = f"""You are {victim}. Respond naturally.
HISTORY: {full_history}
Generate JSON: {{"Name": "{victim}", "Message": "..."}}"""

            msg = generate_turn(prompt)

            if msg is None:
                if turn == 1:
                    msg = {"Name": victim, "Message": "Thanks for reaching out! Could you tell me more about the specific role and team?"}
                elif turn == 3:
                    msg = {"Name": victim, "Message": "That sounds interesting. What would the interview process look like?"}
                elif turn == 5:
                    if persona_type in ["cautious", "formal"]:
                        msg = {"Name": victim, "Message": "I'm happy to share my CV, but I'd prefer to discuss compensation expectations after learning more about the role."}
                    else:
                        msg = {"Name": victim, "Message": "Happy to send my CV! Compensation-wise I'm roughly in the $140-150k range."}
                elif turn == 7:
                    msg = {"Name": victim, "Message": "A call sounds great — I'm free Tuesday or Thursday afternoon."}
                else:
                    msg = {"Name": victim, "Message": "Thanks for the conversation — looking forward to speaking further!"}

        conversation.append(msg)
        history.append(f"{msg['Name']}: {msg['Message']}")

        if turn % 4 == 0:
            torch.cuda.empty_cache()
            time.sleep(0.3)

    return {
        "conversation": conversation,
        "metadata": {
            "conversation_id": f"benign_{config['role'].lower().replace(' ', '_')}_{persona_type}_{conv_num:03d}",
            "conversation_type": "benign",
            "role": config["role"],
            "victim_persona": persona_type,
            "recruiter_style": style_name
        }
    }

# ===================== RUN GENERATION =====================
def run_benign_generation():
    os.makedirs("benign_conversations_v2", exist_ok=True)
    all_results = []


    per_persona = {"naive": 2, "cooperative": 2, "cautious": 1, "formal": 1}
    styles = list(BENIGN_STYLES.keys())
    count = 0

    for config in CONVERSATION_CONFIGS:
        print(f"\n{'='*50}")
        print(f"Scenario: {config['role']}")
        print(f"{'='*50}")
        for persona in ["naive", "cooperative", "cautious", "formal"]:
            for conv_num in range(1, per_persona[persona] + 1):
                style = styles[(conv_num - 1) % len(styles)]
                result = generate_benign_conversation(config, persona, conv_num, style)
                fname = f"benign_conversations_v2/benign_{config['role'].lower().replace(' ', '_')}_{persona}_{conv_num:03d}.json"
                with open(fname, "w") as f:
                    json.dump(result, f, indent=2)
                all_results.append(result)
                count += 1

                # Checkpoint every 10 conversations
                if count % 10 == 0:
                    with zipfile.ZipFile(f'benign_checkpoint_{count}.zip', 'w') as z:
                        for root, dirs, flist in os.walk('benign_conversations_v2'):
                            for file in flist:
                                z.write(os.path.join(root, file))
                    colab_files.download(f'benign_checkpoint_{count}.zip')
                    print(f"  💾 Checkpoint downloaded at {count} conversations")

                time.sleep(1)

    with open("benign_conversations_v2/master.json", "w") as f:
        json.dump(all_results, f, indent=2)

    with open("benign_conversations_v2/summary.csv", "w") as f:
        f.write("conversation_id,conversation_type,role,persona,recruiter_style\n")
        for r in all_results:
            m = r["metadata"]
            f.write(f"{m['conversation_id']},{m['conversation_type']},{m['role']},"
                    f"{m['victim_persona']},{m.get('recruiter_style','')}\n")

    print(f"\n✅ Generated {len(all_results)} benign conversations")

    with zipfile.ZipFile('benign_conversations_v2_final.zip', 'w') as z:
        for root, dirs, flist in os.walk('benign_conversations_v2'):
            for file in flist:
                z.write(os.path.join(root, file))

    colab_files.download('benign_conversations_v2_final.zip')
    print("✅ Final zip downloaded!")

if __name__ == "__main__":
    print("="*60)
    print("BENIGN GENERATOR v2 — 18 per run (run 4x to reach ~66)")
    print("="*60)
    print("✓ Legitimate recruiters mention real timelines")
    print("✓ Legitimate recruiters ask for CV and salary range")
    print("✓ 3 recruiter styles: standard / fast_moving / thorough")
    print("✓ 3 scenarios matching malicious generator")
    print("✓ Cautious/formal victims push back on info requests")
    print("✓ NEVER asks for bank/tax/password — that is the difference")
    print("✓ Checkpoint downloads every 10 conversations")
    print("="*60)
    run_benign_generation()